<a href="https://colab.research.google.com/github/suyon0416/26-2_ESAA_YB/blob/main/ESAA_YB_WEEK13_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 06. 차원 축소 (Dimension Reduction)

> 다루는 알고리즘: **PCA · LDA · SVD · NMF**

**목차**
1. 차원 축소 개요
2. PCA (Principal Component Analysis)
3. LDA (Linear Discriminant Analysis)
4. SVD (Singular Value Decomposition)
5. NMF (Non-Negative Matrix Factorization)
6. 한눈에 비교 & 정리

## 1. 차원 축소 개요

**차원 축소**란 피처가 매우 많은 다차원 데이터를 더 적은 차원의 새로운 데이터로 바꾸는 것이다.

### 왜 필요한가?
| 문제 | 설명 |
|---|---|
| **차원의 저주** | 차원이 커질수록 데이터 포인트 간 거리가 기하급수적으로 멀어지고 데이터가 **희소(sparse)** 해짐 → 모델의 예측 신뢰도 하락 |
| **다중 공선성** | 피처가 많으면 피처끼리 상관관계가 높을 가능성이 큼 → 선형 회귀 같은 선형 모델의 성능 저하 |
| **해석·시각화** | 수십 개 피처는 눈으로 볼 수 없음 → 2~3차원으로 줄이면 시각화 가능 |
| **연산 비용** | 학습 데이터 크기가 줄어 학습에 필요한 처리 능력 감소 |

### 두 가지 방식
- **피처 선택 (Feature Selection)**: 불필요한 피처를 제거하고 데이터를 잘 나타내는 **주요 피처만 골라냄** (기존 피처 그대로 유지)
- **피처 추출 (Feature Extraction)**: 기존 피처를 **새로운 저차원 공간으로 매핑**해 압축 → 기존 피처와 전혀 다른 값이 됨
  - 예) 모의고사·내신·수능·봉사·수상경력 → *학업 성취도, 커뮤니케이션 능력, 문제 해결력*
  - 즉, 원래 피처로는 알기 어려웠던 **잠재 요소(Latent Factor)** 를 뽑아내는 것

### 핵심 포인트 💡
차원 축소의 진짜 의미는 단순 압축이 아니라 **데이터를 더 잘 설명하는 잠재 요소를 찾는 것**이다.
- **이미지**: 픽셀 수가 너무 많아 작은 차이로도 오분류 → 차원 축소로 과적합을 줄여 오히려 성능 향상 가능
- **텍스트**: 문서 속 단어 구성에서 숨은 **시맨틱 토픽(Semantic Topic)** 추출 → SVD, NMF가 토픽 모델링의 기반

## 2. PCA (Principal Component Analysis, 주성분 분석)

### 개념
- 여러 변수 간 **상관관계**를 이용해 이를 대표하는 **주성분(Principal Component)** 을 추출하는 기법
- 정보 유실을 최소화하기 위해 **분산(Variance)이 가장 큰 방향**의 축을 찾는다 (분산 = 데이터의 특성을 가장 잘 나타낸다고 간주)
- 축 생성 순서
  1. 첫 번째 축: 데이터 변동성이 가장 큰 방향
  2. 두 번째 축: 첫 번째 축에 **직교**하면서 그다음으로 변동성이 큰 방향
  3. 세 번째 축: 앞의 축에 직교 … (반복)
- 원본 데이터를 이 축들에 **투영(projection)** 하면 축 개수만큼의 차원으로 축소됨

### 선형대수 관점
PCA = **입력 데이터의 공분산 행렬을 고유값 분해 → 고유벡터로 입력 데이터를 선형 변환**

| 용어 | 의미 |
|---|---|
| 선형 변환 | 벡터에 행렬 $A$를 곱해 새로운 벡터로 바꾸는 것 (다른 공간으로의 투영) |
| 공분산 | 두 변수가 함께 변하는 정도. $Cov(X,Y)>0$ 이면 X가 커질 때 Y도 커짐 |
| 공분산 행렬 | 대각 원소 = 각 변수의 분산, 비대각 원소 = 변수 쌍의 공분산. **정방 + 대칭 행렬** |
| 고유벡터 | 행렬 $A$를 곱해도 **방향은 그대로, 크기만 변하는** 벡터: $Ax = \lambda x$ |
| 고유값 | 고유벡터의 크기 = 그 방향의 **분산 크기** |

공분산 행렬 $C$는 대칭행렬이므로 항상 **직교행렬 $P$** 와 **대각행렬 $\Sigma$** 로 대각화할 수 있다.

$$C = P\,\Sigma\,P^{T} = [e_1 \cdots e_n]\begin{bmatrix}\lambda_1 & & \\ & \ddots & \\ & & \lambda_n\end{bmatrix}\begin{bmatrix}e_1^T \\ \vdots \\ e_n^T\end{bmatrix}$$

- $e_1$: 분산이 가장 큰 방향의 고유벡터, $e_2$: $e_1$에 수직이면서 다음으로 분산이 큰 방향, …
- $\lambda_i$: $i$번째 고유벡터의 크기(분산)

### PCA 수행 단계
1. 입력 데이터의 **공분산 행렬** 생성
2. 공분산 행렬의 **고유벡터·고유값** 계산
3. 고유값이 큰 순서로 **K개**(변환할 차원 수) 고유벡터 추출
4. 추출한 고유벡터로 입력 데이터를 **변환**

### 2-1. 붓꽃(Iris) 데이터 로딩
4개 속성(sepal length, sepal width, petal length, petal width)을 2개의 PCA 차원으로 압축해 본다.

In [ ]:
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

iris = load_iris()
# 넘파이 데이터 세트를 판다스 DataFrame으로 변환
columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
irisDF = pd.DataFrame(iris.data, columns=columns)
irisDF['target'] = iris.target
irisDF.head(3)

,sepal_length,sepal_width,petal_length,petal_width,target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0


### 2-2. 원본 데이터 분포 시각화 (sepal length × sepal width)

In [ ]:
#setosa는 세모, versicolor는 네모, virginica는 동그라미로 표현
markers = ['^', 's', 'o']

#setosa의 target 값은 0, versicolor는 1, virginica는 2. 각 target별로 다른 모양으로 산점도로 표시
for i, marker in enumerate(markers):
    x_axis_data = irisDF[irisDF['target']==i]['sepal_length']
    y_axis_data = irisDF[irisDF['target']==i]['sepal_width']
    plt.scatter(x_axis_data, y_axis_data, marker=marker, label=iris.target_names[i])

plt.legend()
plt.xlabel('sepal length')
plt.ylabel('sepal width')
plt.show()

- setosa는 sepal width > 3.0, sepal length ≤ 6.0 영역에 모여 있어 구분이 쉽다.
- versicolor와 virginica는 sepal 두 속성만으로는 구분이 어렵다.

### 2-3. 스케일링 → PCA 변환
> ⚠️ PCA는 여러 속성 값을 함께 연산하므로 **스케일에 영향을 받는다** → 적용 전 `StandardScaler`로 평균 0, 분산 1로 표준화한다.

- `PCA(n_components=2)`: 변환할 차원 수
- `fit()` → `transform()` 으로 변환

In [ ]:
from sklearn.preprocessing import StandardScaler

# Target 값을 제외한 모든 속성 값을 StandardScaler를 이용해 표준 정규 분포를 가지는 값들로 변환
iris_scaled = StandardScaler().fit_transform(irisDF.iloc[:, :-1])

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)

#fit( )과 transform( ) 을 호출해 PCA 변환 데이터 반환
pca.fit(iris_scaled)
iris_pca = pca.transform(iris_scaled)
print(iris_pca.shape)

(150, 2)


In [ ]:
# PCA 변환된 데이터의 칼럼명을 각각 pca_component_1, pca_component_2로 명명
pca_columns = ['pca_component_1', 'pca_component_2']
irisDF_pca = pd.DataFrame(iris_pca, columns=pca_columns)
irisDF_pca['target'] = iris.target
irisDF_pca.head(3)

,pca_component_1,pca_component_2,target
0,-2.264703,0.480027,0
1,-2.080961,-0.674134,0
2,-2.364229,-0.341908,0


In [ ]:
#setosa를 세모, versicolor를 네모, virginica를 동그라미로 표시
markers = ['^', 's', 'o']

#pca_component_1 을 x축, pc_component_2를 y축으로 scatter plot 수행.
for i, marker in enumerate(markers):
    x_axis_data = irisDF_pca[irisDF_pca['target']==i]['pca_component_1']
    y_axis_data = irisDF_pca[irisDF_pca['target']==i]['pca_component_2']
    plt.scatter(x_axis_data, y_axis_data, marker=marker, label=iris.target_names[i])

plt.legend()
plt.xlabel('pca_component_1')
plt.ylabel('pca_component_2')
plt.show()

- PCA 변환 후 `pca_component_1` 축만으로도 setosa가 명확히 구분되고, versicolor/virginica도 일부 겹치지만 비교적 잘 나뉜다.
- 첫 번째 주성분이 원본 데이터의 변동성을 잘 반영했기 때문이다.

### 2-4. 주성분별 변동성 비율 `explained_variance_ratio_`
전체 변동성에서 각 PCA 컴포넌트가 차지하는 비율

In [ ]:
print(pca.explained_variance_ratio_)

[0.72962445 0.22850762]


→ PC1 ≈ **72.9%**, PC2 ≈ **22.8%** → 2개 성분만으로 원본 변동성의 약 **95%** 설명

### 2-5. 원본 vs PCA 변환 데이터 분류 성능 비교 (RandomForest, 3-fold CV)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

rcf = RandomForestClassifier(random_state=156)
scores = cross_val_score(rcf, iris.data, iris.target, scoring='accuracy', cv=3)
print('원본 데이터 교차 검증 개별 정확도:', scores)
print('원본 데이터 평균 정확도:', np.mean(scores))

원본 데이터 교차 검증 개별 정확도: [0.98 0.94 0.96]
원본 데이터 평균 정확도: 0.96


In [ ]:
pca_X = irisDF_pca[['pca_component_1', 'pca_component_2']]
scores_pca = cross_val_score(rcf, pca_X, iris.target, scoring='accuracy', cv=3)
print('PCA 변환 데이터 교차 검증 개별 정확도:', scores_pca)
print('PCA 변환 데이터 평균 정확도:', np.mean(scores_pca))

PCA 변환 데이터 교차 검증 개별 정확도: [0.88 0.88 0.88]
PCA 변환 데이터 평균 정확도: 0.88


- 교재 결과: 원본 **0.96** → PCA **0.88** (약 8% 하락)
- 정확도는 떨어졌지만 속성 수를 **50%** 줄였는데도 원본 특성을 상당 부분 유지한다.

### 2-6. 신용카드 고객 데이터 세트 (피처 23개)
- UCI *Default of Credit Card Clients* (30,000건, 24개 속성, 타깃 `default payment next month` = 다음 달 연체 여부: 연체 1 / 정상 0)
- 교재는 xls 파일을 직접 내려받아 `pca_credit_card.xls`로 저장한다.

> 📥 **(Colab 준비용 셀 — 교재 외)** Colab에서 교재 코드를 그대로 실행할 수 있도록 UCI에서 파일을 받아 `pca_credit_card.xls`로 저장한다. (xls 읽기에 `xlrd` 필요)

In [ ]:
!pip -q install xlrd
import urllib.request, zipfile, io, shutil

url = 'https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip'
with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())) as z:
    xls_name = [n for n in z.namelist() if n.endswith('.xls')][0]
    with z.open(xls_name) as src, open('pca_credit_card.xls', 'wb') as dst:
        shutil.copyfileobj(src, dst)
print('saved: pca_credit_card.xls')


[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


saved: pca_credit_card.xls


In [ ]:
# header로 의미없는 첫행 제거, iloc로 기존 id 제거
import pandas as pd

df = pd.read_excel('pca_credit_card.xls', header=1, sheet_name='Data').iloc[0:, 1:]
print(df.shape)
df.head(3)

(30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0


- `PAY_0` 다음이 `PAY_2`이므로 `PAY_0` → `PAY_1`로, 긴 타깃 칼럼명은 `default`로 변경

In [ ]:
df.rename(columns={'PAY_0':'PAY_1', 'default payment next month':'default'}, inplace=True)
y_target = df['default']
X_features = df.drop('default', axis=1)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

corr = X_features.corr()
plt.figure(figsize=(14, 14))
sns.heatmap(corr, annot=True, fmt='.1g')

- `BILL_AMT1 ~ BILL_AMT6` 간 상관도는 대부분 **0.9 이상**, `PAY_1 ~ PAY_6` 도 상관도가 높다.
- 상관도가 높은 속성들은 **소수의 PCA 성분만으로 변동성을 자연스럽게 수용**할 수 있다.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# BILL_AMT1 ~ BILL_AMT6까지 6개의 속성명 생성
cols_bill = ['BILL_AMT'+str(i) for i in range(1, 7)]
print('대상 속성명:', cols_bill)

# 2개의 PCA 속성을 가진 PCA 객체 생성하고, explained_variance_ratio_ 계산을 위해 fit( ) 호출
scaler = StandardScaler()
df_cols_scaled = scaler.fit_transform(X_features[cols_bill])
pca = PCA(n_components=2)
pca.fit(df_cols_scaled)
print('PCA Component별 변동성:', pca.explained_variance_ratio_)

대상 속성명: ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
PCA Component별 변동성: [0.90555253 0.0509867 ]


→ 2개 성분으로 약 **95%**, 첫 번째 성분 하나로 약 **90%** 설명 → 6개 속성의 상관도가 매우 높다는 뜻

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rcf = RandomForestClassifier(n_estimators=300, random_state=156)
scores = cross_val_score(rcf, X_features, y_target, scoring='accuracy', cv=3)

print('CV=3 인 경우의 개별 Fold세트별 정확도:', scores)
print('평균 정확도:{0:.4f}'.format(np.mean(scores)))

CV=3 인 경우의 개별 Fold세트별 정확도: [0.8083 0.8196 0.8232]
평균 정확도:0.8170


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 원본 데이터셋에 먼저 StandardScaler적용
scaler = StandardScaler()
df_scaled = scaler.fit_transform(X_features)

# 6개의 Component를 가진 PCA 변환을 수행하고 cross_val_score( )로 분류 예측 수행.
pca = PCA(n_components=6)
df_pca = pca.fit_transform(df_scaled)
scores_pca = cross_val_score(rcf, df_pca, y_target, scoring='accuracy', cv=3)

print('CV=3 인 경우의 PCA 변환된 개별 Fold세트별 정확도:', scores_pca)
print('PCA 변환 데이터 셋 평균 정확도:{0:.4f}'.format(np.mean(scores_pca)))

CV=3 인 경우의 PCA 변환된 개별 Fold세트별 정확도: [0.7913 0.7972 0.802 ]
PCA 변환 데이터 셋 평균 정확도:0.7968


- 교재 결과: 원본 **0.8170** vs PCA 6개 성분 **0.7973**
- 전체 23개 속성의 약 **1/4** 만으로 성능 저하가 1~2% 수준 → PCA의 뛰어난 압축 능력
- PCA는 특히 **컴퓨터 비전**(얼굴 인식의 *Eigen-face*) 분야에서 많이 활용된다.

## 3. LDA (Linear Discriminant Analysis, 선형 판별 분석)

### 개념
- PCA처럼 데이터를 저차원 공간에 투영해 차원을 줄이지만, **목표가 다르다.**
  - PCA: 입력 데이터의 **변동성(분산)이 가장 큰 축**을 찾음 → **비지도 학습**
  - LDA: 타깃 **클래스를 최대한 분리할 수 있는 축**을 찾음 → **지도 학습** (변환 시 결정값 `y` 필요!)

### 원리
**클래스 간 분산(between-class scatter)은 최대한 크게, 클래스 내부 분산(within-class scatter)은 최대한 작게** → 이 비율을 최대화

$$\text{maximize}\;\; \frac{\text{클래스 간 분산 } S_B}{\text{클래스 내부 분산 } S_W}$$

### LDA 수행 단계
1. 클래스별 피처 **평균 벡터**를 이용해 클래스 내부 분산 행렬 $S_W$, 클래스 간 분산 행렬 $S_B$ 계산
2. 두 행렬을 고유벡터로 분해: $S_W^{-1} S_B = [e_1 \cdots e_n]\,\mathrm{diag}(\lambda_1,\dots,\lambda_n)\,[e_1 \cdots e_n]^T$
3. 고유값이 큰 순서로 **K개**(LDA 변환 차수) 추출
4. 추출한 고유벡터로 입력 데이터 변환

> PCA와 단계는 거의 같고, **공분산 행렬 대신 $S_W, S_B$ 를 쓴다**는 것이 핵심 차이.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

iris = load_iris()
iris_scaled = StandardScaler().fit_transform(iris.data)

- ⚠️ `fit()` 호출 시 **결정값(`iris.target`)을 함께 입력**한다 (지도학습)

In [ ]:
lda = LinearDiscriminantAnalysis(n_components=2)
lda.fit(iris_scaled, iris.target)
iris_lda = lda.transform(iris_scaled)
print(iris_lda.shape)

(150, 2)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

lda_columns = ['lda_component_1', 'lda_component_2']
irisDF_lda = pd.DataFrame(iris_lda, columns=lda_columns)
irisDF_lda['target'] = iris.target

#setosa는 세모, versicolor는 네모, virginica는 동그라미로 표현
markers = ['^', 's', 'o']

#setosa의 target 값은 0, versicolor는 1, virginica는 2. 각 target 별로 다른 shape으로 scatter plot
for i, marker in enumerate(markers):
    x_axis_data = irisDF_lda[irisDF_lda['target']==i]['lda_component_1']
    y_axis_data = irisDF_lda[irisDF_lda['target']==i]['lda_component_2']
    plt.scatter(x_axis_data, y_axis_data, marker=marker, label=iris.target_names[i])

plt.legend(loc='upper right')
plt.xlabel('lda_component_1')
plt.ylabel('lda_component_2')
plt.show()

- LDA로 변환한 결과는 PCA 결과와 **좌우 대칭 형태로 많이 닮아 있다**.

## 4. SVD (Singular Value Decomposition, 특이값 분해)

### 개념
- PCA의 고유값 분해는 **정방행렬**에만 적용되지만, SVD는 **행과 열의 크기가 다른 행렬(m×n)** 에도 적용 가능

$$A = U\,\Sigma\,V^{T}$$

| 행렬 | 크기 | 의미 |
|---|---|---|
| $U$ | m × m | 왼쪽 특이벡터 (서로 직교) |
| $\Sigma$ | m × n | **대각행렬**. 대각 원소만 0이 아니며 이것이 **특이값** |
| $V^T$ | n × n | 오른쪽 특이벡터 (서로 직교) |

### SVD의 종류
- **Full SVD**: 위의 원래 형태
- **Compact SVD**: $\Sigma$의 비대각 부분과 특이값 0인 부분을 제거하고 대응하는 $U, V$ 원소도 제거 → $U: m\times p,\ \Sigma: p\times p,\ V^T: p\times n$
- **Truncated SVD**: 특이값 중 **상위 몇 개만** 남기고 대응하는 $U, V$ 원소도 제거 → 원본을 정확히 복원할 수는 없지만 **근사**할 수 있음 (원래 차수에 가깝게 자를수록 더 정확)

### 4-1. 넘파이 SVD로 분해·복원 (`numpy.linalg.svd`)
랜덤 행렬을 쓰는 이유: 행렬의 개별 로우끼리의 **의존성을 없애기 위해**

In [ ]:
# numpy의 svd 모듈 import
import numpy as np
from numpy.linalg import svd

# 4X4 Random 행렬 a 생성
np.random.seed(121)
a = np.random.randn(4, 4)
print(np.round(a, 3))

[[-0.212 -0.285 -0.574 -0.44 ]
 [-0.33   1.184  1.615  0.367]
 [-0.014  0.63   1.71  -1.327]
 [ 0.402 -0.191  1.404 -1.969]]


- `svd()`는 $U$, $\Sigma$, $V^T$를 반환하며, $\Sigma$는 **0이 아닌 대각값만 1차원 배열**로 반환한다.

In [ ]:
U, Sigma, Vt = svd(a)
print(U.shape, Sigma.shape, Vt.shape)
print('U matrix:\n', np.round(U, 3))
print('Sigma Value:\n', np.round(Sigma, 3))
print('V transpose matrix:\n', np.round(Vt, 3))

(4, 4) (4,) (4, 4)
U matrix:
 [[-0.079 -0.318  0.867  0.376]
 [ 0.383  0.787  0.12   0.469]
 [ 0.656  0.022  0.357 -0.664]
 [ 0.645 -0.529 -0.328  0.444]]
Sigma Value:
 [3.423 2.023 0.463 0.079]
V transpose matrix:
 [[ 0.041  0.224  0.786 -0.574]
 [-0.2    0.562  0.37   0.712]
 [-0.778  0.395 -0.333 -0.357]
 [-0.593 -0.692  0.366  0.189]]


- 복원 = $U \cdot \Sigma \cdot V^T$ 내적. 단, Sigma를 **다시 0을 포함한 대각행렬로 변환**한 뒤 내적해야 한다.

In [ ]:
# Sima를 다시 0 을 포함한 대칭행렬로 변환
Sigma_mat = np.diag(Sigma)
a_ = np.dot(np.dot(U, Sigma_mat), Vt)
print(np.round(a_, 3))

[[-0.212 -0.285 -0.574 -0.44 ]
 [-0.33   1.184  1.615  0.367]
 [-0.014  0.63   1.71  -1.327]
 [ 0.402 -0.191  1.404 -1.969]]


### 4-2. 로우 간 의존성이 있는 경우
3번째 로우 = 1번째 + 2번째 로우, 4번째 로우 = 1번째 로우로 바꿔 일부러 의존성을 부여한다.

In [ ]:
a[2] = a[0] + a[1]
a[3] = a[0]
print(np.round(a, 3))

[[-0.212 -0.285 -0.574 -0.44 ]
 [-0.33   1.184  1.615  0.367]
 [-0.542  0.899  1.041 -0.073]
 [-0.212 -0.285 -0.574 -0.44 ]]


In [ ]:
# 다시 SVD를 수행하여 Sigma 값 확인
U, Sigma, Vt = svd(a)
print(U.shape, Sigma.shape, Vt.shape)
print('Sigma Value:\n', np.round(Sigma, 3))

(4, 4) (4,) (4, 4)
Sigma Value:
 [2.663 0.807 0.    0.   ]


- Sigma 값 중 2개가 0 → 선형 독립인 로우 벡터가 2개, 즉 **행렬의 랭크(Rank) = 2**
- 0이 아닌 특이값 2개에 대응하는 부분만 사용해 복원해 본다: $U$는 앞 2열, $V^T$는 앞 2행

In [ ]:
# U 행렬의 경우는 Sigma와 내적을 수행하므로 Sigma의 앞 2행에 대응되는 앞 2열만 추출
U_ = U[:, :2]
Sigma_ = np.diag(Sigma[:2])
# V 전치 행렬의 경우는 앞 2행만 추출
Vt_ = Vt[:2]
print(U_.shape, Sigma_.shape, Vt_.shape)
# U, Sigma, Vt의 내적을 수행하며, 다시 원본 행렬 복원
a_ = np.dot(np.dot(U_, Sigma_), Vt_)
print(np.round(a_, 3))

(4, 2) (2, 2) (2, 4)
[[-0.212 -0.285 -0.574 -0.44 ]
 [-0.33   1.184  1.615  0.367]
 [-0.542  0.899  1.041 -0.073]
 [-0.212 -0.285 -0.574 -0.44 ]]


→ 2개 성분만으로도 원본이 그대로 복원된다.

### 4-3. Truncated SVD (`scipy`)
- 일반 SVD: `scipy.linalg.svd`
- Truncated SVD: **희소 행렬로만 지원** → `scipy.sparse.linalg.svds` (넘파이에서는 지원 X)

In [ ]:
import numpy as np
from scipy.sparse.linalg import svds
from scipy.linalg import svd

# 원본 행렬을 출력하고, SVD를 적용할 경우 U, Sigma, Vt 의 차원 확인
np.random.seed(121)
matrix = np.random.random((6, 6))
print('원본 행렬:\n', matrix)
U, Sigma, Vt = svd(matrix, full_matrices=False)
print('\n분해 행렬 차원:', U.shape, Sigma.shape, Vt.shape)
print('\nSigma값 행렬:', Sigma)

# Truncated SVD로 Sigma 행렬의 특이값을 4개로 하여 Truncated SVD 수행.
num_components = 4
U_tr, Sigma_tr, Vt_tr = svds(matrix, k=num_components)
print('\nTruncated SVD 분해 행렬 차원:', U_tr.shape, Sigma_tr.shape, Vt_tr.shape)
print('\nTruncated SVD Sigma값 행렬:', Sigma_tr)
matrix_tr = np.dot(np.dot(U_tr, np.diag(Sigma_tr)), Vt_tr)  # output of TruncatedSVD

print('\nTruncated SVD로 분해 후 복원 행렬:\n', matrix_tr)

원본 행렬:
 [[0.11133083 0.21076757 0.23296249 0.15194456 0.83017814 0.40791941]
 [0.5557906  0.74552394 0.24849976 0.9686594  0.95268418 0.48984885]
 [0.01829731 0.85760612 0.40493829 0.62247394 0.29537149 0.92958852]
 [0.4056155  0.56730065 0.24575605 0.22573721 0.03827786 0.58098021]
 [0.82925331 0.77326256 0.94693849 0.73632338 0.67328275 0.74517176]
 [0.51161442 0.46920965 0.6439515  0.82081228 0.14548493 0.01806415]]

분해 행렬 차원: (6, 6) (6,) (6, 6)

Sigma값 행렬: [3.2535007  0.88116505 0.83865238 0.55463089 0.35834824 0.0349925 ]



Truncated SVD 분해 행렬 차원: (6, 4) (4,) (4, 6)

Truncated SVD Sigma값 행렬: [0.55463089 0.83865238 0.88116505 3.2535007 ]

Truncated SVD로 분해 후 복원 행렬:
 [[0.19222941 0.21792946 0.15951023 0.14084013 0.81641405 0.42533093]
 [0.44874275 0.72204422 0.34594106 0.99148577 0.96866325 0.4754868 ]
 [0.12656662 0.88860729 0.30625735 0.59517439 0.28036734 0.93961948]
 [0.23989012 0.51026588 0.39697353 0.27308905 0.05971563 0.57156395]
 [0.83806144 0.78847467 0.93868685 0.72673231 0.6740867  0.73812389]
 [0.59726589 0.47953891 0.56613544 0.80746028 0.13135039 0.03479656]]


- 일반 SVD: (6,6) (6,) (6,6) → Truncated SVD(k=4): (6,4) (4,) (4,6)
- Truncated SVD로 복원하면 **완벽하지 않고 근사적으로** 복원된다.

### 4-4. 사이킷런 `TruncatedSVD` 클래스
- scipy `svds`처럼 $U, \Sigma, V^T$를 반환하지 않고, PCA처럼 `fit()` / `transform()`으로 원본 데이터를 **K개 컴포넌트로 차원 축소**한다.
- 원본 데이터를 Truncated SVD로 분해된 $U \cdot \Sigma$ 행렬에 선형 변환해 생성

In [ ]:
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
%matplotlib inline

iris = load_iris()
iris_ftrs = iris.data
# 2개의 주요 component로 TruncatedSVD 변환
tsvd = TruncatedSVD(n_components=2)
tsvd.fit(iris_ftrs)
iris_tsvd = tsvd.transform(iris_ftrs)

# Scatter plot 2차원으로 TruncatedSVD 변환 된 데이터 표현. 품종은 색깔로 구분
plt.scatter(x=iris_tsvd[:, 0], y=iris_tsvd[:, 1], c=iris.target)
plt.xlabel('TruncatedSVD Component 1')
plt.ylabel('TruncatedSVD Component 2')

- TruncatedSVD 변환 후에도 품종별로 어느 정도 클러스터링이 가능할 만큼 변환 속성이 뛰어난 고유성을 가진다.
- 데이터를 스케일링한 뒤 TruncatedSVD와 PCA를 비교하면 **거의 동일**하다.

In [ ]:
from sklearn.preprocessing import StandardScaler

# iris 데이터를 StandardScaler로 변환
scaler = StandardScaler()
iris_scaled = scaler.fit_transform(iris_ftrs)

# 스케일링된 데이터를 기반으로 TruncatedSVD 변환 수행
tsvd = TruncatedSVD(n_components=2)
tsvd.fit(iris_scaled)
iris_tsvd = tsvd.transform(iris_scaled)

# 스케일링된 데이터를 기반으로 PCA 변환 수행
pca = PCA(n_components=2)
pca.fit(iris_scaled)
iris_pca = pca.transform(iris_scaled)

# TruncatedSVD 변환 데이터를 왼쪽에, PCA변환 데이터를 오른쪽에 표현
fig, (ax1, ax2) = plt.subplots(figsize=(9, 4), ncols=2)
ax1.scatter(x=iris_tsvd[:, 0], y=iris_tsvd[:, 1], c=iris.target)
ax2.scatter(x=iris_pca[:, 0], y=iris_pca[:, 1], c=iris.target)
ax1.set_title('Truncated SVD Transformed')
ax2.set_title('PCA Transformed')

In [ ]:
print((iris_pca - iris_tsvd).mean())
print((pca.components_ - tsvd.components_).mean())

2.3440740089715935e-15
2.0816681711721685e-17


- 두 값 모두 0에 가까움 → **스케일링으로 데이터 중심이 같아지면 사이킷런 SVD와 PCA는 동일한 변환**을 수행한다 (= PCA가 SVD로 구현되어 있음).
- 차이점: **PCA는 밀집 행렬(Dense Matrix)만**, **SVD는 희소 행렬(Sparse Matrix)도** 변환 가능
- 활용: 이미지 압축·패턴 인식, 신호 처리, 텍스트 토픽 모델링 **LSA(Latent Semantic Analysis)** 의 기반 알고리즘

## 5. NMF (Non-Negative Matrix Factorization, 비음수 행렬 분해)

### 개념
- Truncated SVD처럼 **낮은 랭크를 통한 행렬 근사(Low-Rank Approximation)** 방식의 변형
- 원본 행렬의 모든 원소가 **0 이상(양수)** 임이 보장되면, 두 개의 **기반 양수 행렬**로 분해

$$V_{(4\times 6)} \approx W_{(4\times 2)}\; H_{(2\times 6)}$$

| 행렬 | 형태 | 의미 |
|---|---|---|
| $W$ | 길고 가는 행렬 (행 크기 = 원본 행, 열 크기 < 원본 열) | **원본 행(샘플)** 에 잠재 요소 값이 얼마나 되는지 |
| $H$ | 작고 넓은 행렬 (행 크기 < 원본 행, 열 크기 = 원본 열) | **잠재 요소가 원본 열(속성)** 로 어떻게 구성되는지 |

- **행렬 분해(Matrix Factorization)**: SVD 같은 행렬 분해 기법을 통칭하는 말

In [ ]:
from sklearn.decomposition import NMF
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt
%matplotlib inline

iris = load_iris()
iris_ftrs = iris.data
nmf = NMF(n_components=2)
nmf.fit(iris_ftrs)
iris_nmf = nmf.transform(iris_ftrs)
plt.scatter(x=iris_nmf[:, 0], y=iris_nmf[:, 1], c=iris.target)
plt.xlabel('NMF Component 1')
plt.ylabel('NMF Component 2')

C:\Users\정수연\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


- 활용: 이미지 압축·패턴 인식, 텍스트 **토픽 모델링**, 문서 유사도·클러스터링
- 특히 **추천 시스템**: 사용자-평가 순위(User-Rating) 행렬을 분해해 사용자가 평가하지 않은 상품의 잠재 요소를 추출 → 평점 예측 → 높은 순위 상품 추천
  → **잠재 요소(Latent Factoring) 기반 추천**

## 6. 한눈에 비교 & 정리

| | PCA | LDA | SVD | NMF |
|---|---|---|---|---|
| 학습 방식 | 비지도 | **지도** (레이블 필요) | 비지도 | 비지도 |
| 찾는 것 | 분산이 가장 큰 축 | 클래스를 가장 잘 나누는 축 | 특이값/특이벡터 | 음수 없는 두 행렬 W, H |
| 분해 대상 | 공분산 행렬 (고유값 분해) | $S_W^{-1}S_B$ (고유값 분해) | 임의의 m×n 행렬 | 원소가 0 이상인 행렬 |
| 수식 | $C = P\Sigma P^T$ | $\max S_B / S_W$ | $A = U\Sigma V^T$ | $V \approx WH$ |
| 사이킷런 | `PCA` | `LinearDiscriminantAnalysis` | `TruncatedSVD` | `NMF` |
| 주 활용 | 시각화, 압축, Eigen-face | 분류 전처리 | 희소 행렬, LSA, 이미지 | 토픽 모델링, 추천 |

### 핵심 요약 ✍️
1. 차원 축소의 본질은 피처 수를 줄이는 것보다 **데이터를 잘 설명하는 잠재 요소를 추출**하는 데 있다.
2. **PCA**: 공분산 행렬의 고유벡터(분산이 큰 방향)로 데이터를 투영. 적용 전 **스케일링 필수**.
3. **LDA**: PCA와 비슷하지만, 분산 대신 **클래스 분리도**(클래스 간 분산↑, 클래스 내 분산↓)를 최대화하는 지도 학습 기법.
4. **SVD**: 정방행렬이 아니어도 분해 가능. Truncated SVD로 근사 분해하며, 스케일링된 데이터에서는 PCA와 동일. 희소 행렬 지원.
5. **NMF**: 음수가 없는 행렬을 두 양수 행렬로 근사 분해. 토픽 모델링·추천 시스템에서 활발히 사용.